# Random Variables and Distributions

| Difficulty | ███░░░░░░░ 3/10 |
| :--- | :--- |
| Prerequisites | 03.01 |
| Time | ~60 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/03_Probability_and_Statistics/02_random_variables_distributions.ipynb)

---

## 🎯 Learning Objective

Distinguish discrete and continuous random variables and learn the PMF, PDF, CDF, and quantile function that describe them.

---

## 📐 Theory

A **random variable** $X$ is a function that maps outcomes in a sample space $\Omega$ (`03.01`)
to real numbers: $X:\Omega\to\mathbb{R}$. It lets us stop talking about abstract outcomes ("the
die shows a 4") and start talking about numbers we can compute with ("$X=4$"). Every random
variable is either discrete, continuous, or a mix — this notebook covers the tools for
describing each.

### Discrete random variables and the PMF

$X$ is **discrete** if it takes values in a finite or countable set. Its **probability mass
function (PMF)** is $p_X(x) = P(X=x)$, and validity requires exactly the axioms from `03.01`:
$p_X(x)\ge 0$ for every $x$, and $\sum_x p_X(x) = 1$ (normalization, summed over the support).
Rolling a fair die gives $p_X(x) = 1/6$ for $x\in\{1,\ldots,6\}$.

### Continuous random variables and the PDF

$X$ is **continuous** if it can take any value in an interval of reals. Its **probability
density function (PDF)** $f_X(x)$ satisfies

$$P(a \le X \le b) = \int_a^b f_X(x)\,dx, \qquad f_X(x) \ge 0, \qquad \int_{-\infty}^{\infty} f_X(x)\,dx = 1.$$

The crucial subtlety: $f_X(x)$ is **not** a probability. It's a *density* — probability per
unit length — and can exceed 1 (a concentrated distribution has a tall, narrow PDF). Only after
integrating over an interval do you get an actual probability. A direct consequence:
$P(X=x)=\int_x^x f_X(t)\,dt=0$ for any single point $x$ — only intervals carry probability under
a continuous distribution.

### The CDF: one tool for both cases

The **cumulative distribution function (CDF)** $F_X(x) = P(X \le x)$ is defined identically for
discrete and continuous random variables, which is exactly why it's the universal tool. It is
always non-decreasing, right-continuous, with $F_X(-\infty)=0$ and $F_X(\infty)=1$. For a
discrete $X$, $F_X$ is a **step function**: flat everywhere except jumps of height $p_X(x)$ at
each point in the support. For a continuous $X$,

$$F_X(x) = \int_{-\infty}^x f_X(t)\,dt, \qquad\text{so}\qquad f_X(x) = F_X'(x)$$

wherever $F_X$ is differentiable — the PDF is literally the derivative of the CDF, the same
Fundamental Theorem of Calculus relationship between accumulated area and rate from `02.07`
(link: [Integration Fundamentals](../02_Calculus/07_integration_fundamentals.ipynb)), applied to
probability instead of area under an arbitrary curve.

### Quantiles: inverting the CDF

The **quantile function** (inverse CDF) answers the opposite question from the CDF: instead of
"what's the probability of being below $x$?" it asks "what value has cumulative probability
$q$?"

$$Q(q) = F_X^{-1}(q) = \inf\{x : F_X(x) \ge q\}, \qquad q \in (0,1).$$

$Q(0.5)$ is the median; $Q(0.25)$ and $Q(0.75)$ are the quartiles. Quantiles matter beyond
reporting percentiles: if $U\sim$Uniform$(0,1)$, then $Q(U)$ has exactly distribution $F_X$ — a
fact called the **inverse transform**, the basis of a simple general-purpose sampling
algorithm, built fully in [Sampling and Monte Carlo Methods](09_sampling_and_monte_carlo.ipynb) (`03.09`).

### The `scipy.stats` pattern

Every distribution in `scipy.stats` exposes the same four methods, and this notebook
establishes the pattern used throughout the rest of this module: `.pmf(x)`/`.pdf(x)` (mass or
density at $x$), `.cdf(x)` ($P(X\le x)$), `.ppf(q)` (the quantile function $Q(q)$, "percent
point function"), and `.rvs(size=n)` (draw $n$ random samples). Once you know these four names,
you know how to use *any* of the dozens of distributions surveyed in
[A Gallery of Common Distributions](06_common_distributions_gallery.ipynb) (`03.06`).

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, binom, expon
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Side by side: a discrete PMF/CDF pair (binomial) and a continuous PDF/CDF pair (Gaussian). The
CDF is always the "running total" of the PMF/PDF to its left — a step-accumulation for the
discrete case, a smooth accumulation of area for the continuous case. Quantiles then read the
CDF plot *backward*: pick a probability on the y-axis, read off the x-value.

In [ ]:
# The scipy.stats pattern, used identically for a discrete and a continuous distribution:
# .pmf/.pdf (mass/density), .cdf (cumulative probability), .ppf (quantile, inverse CDF), .rvs (samples)
n, p_success = 10, 0.3
binom_rv = binom(n, p_success)
k_vals = np.arange(0, n + 1)

mu, sigma = 0.0, 1.0
normal_rv = norm(mu, sigma)
x_vals = np.linspace(-4, 4, 400)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].bar(k_vals, binom_rv.pmf(k_vals), color="#4C72B0")
axes[0, 0].set_title(f"Discrete PMF: Binomial(n={n}, p={p_success})")
axes[0, 0].set_xlabel("k"); axes[0, 0].set_ylabel("P(X=k)")

axes[0, 1].step(k_vals, binom_rv.cdf(k_vals), where="post", color="#4C72B0", lw=2)
axes[0, 1].set_title("Discrete CDF: a step function\n(jump height = PMF at that point)")
axes[0, 1].set_xlabel("k"); axes[0, 1].set_ylabel("P(X<=k)")

axes[1, 0].plot(x_vals, normal_rv.pdf(x_vals), color="#C44E52", lw=2)
fill_region = (x_vals >= -1) & (x_vals <= 1)
axes[1, 0].fill_between(x_vals[fill_region], normal_rv.pdf(x_vals[fill_region]), color="#C44E52", alpha=0.3)
axes[1, 0].set_title(f"Continuous PDF: N({mu},{sigma})\nshaded area = P(-1<=X<=1) = {normal_rv.cdf(1)-normal_rv.cdf(-1):.4f}")
axes[1, 0].set_xlabel("x"); axes[1, 0].set_ylabel("f(x) [a density, not a probability]")

axes[1, 1].plot(x_vals, normal_rv.cdf(x_vals), color="#C44E52", lw=2)
quantile_probs = [0.25, 0.5, 0.75]
quantile_xs = normal_rv.ppf(quantile_probs)   # the inverse CDF: .ppf reads the CDF "backward"
for q_prob, q_x in zip(quantile_probs, quantile_xs):
    axes[1, 1].plot([q_x, q_x, -4], [0, q_prob, q_prob], "k--", lw=1, alpha=0.6)
    axes[1, 1].plot(q_x, q_prob, "ko", markersize=5)
axes[1, 1].set_title("Continuous CDF + quantiles (.ppf)\ndashed lines: read CDF backward from y to x")
axes[1, 1].set_xlabel("x"); axes[1, 1].set_ylabel("P(X<=x)")

plt.tight_layout()
plt.show()

print(f"PMF sums to 1 over the support: {binom_rv.pmf(k_vals).sum():.10f}")
print(f"PDF at its peak (x=0) is f(0)={normal_rv.pdf(0):.4f} -- a density can exceed 1; only its integral cannot")
print(f"Quantiles (.ppf) at 0.25/0.5/0.75: {np.round(quantile_xs, 4)} -- the median (.ppf(0.5)) is {quantile_xs[1]:.4f}, as expected for a symmetric N(0,1)")
print(f".rvs(5) draws actual samples: {np.round(normal_rv.rvs(size=5, random_state=0), 3)}")

## 🐍 Implementation from Scratch

We build a discrete PMF/CDF directly from raw counts, then implement a generic quantile
function by numerically inverting *any* CDF via bisection, cross-checking it against
`scipy.stats.norm.ppf`. Finally we use our from-scratch quantile function to implement inverse
transform sampling and confirm the samples follow the target distribution.

In [ ]:
# A discrete PMF/CDF built directly from raw observation counts (e.g. a possibly-loaded die)
raw_counts = np.array([10, 15, 20, 25, 5, 25])   # 100 rolls of a die, faces 1-6
support = np.arange(1, 7)
pmf_from_scratch = raw_counts / raw_counts.sum()
cdf_from_scratch = np.cumsum(pmf_from_scratch)

print("Empirical PMF from raw counts:", np.round(pmf_from_scratch, 3))
print(f"PMF sums to 1: {pmf_from_scratch.sum():.10f}")
print("Empirical CDF (running total of the PMF):", np.round(cdf_from_scratch, 3))


def quantile_from_cdf(cdf_func, target_prob, lo, hi, tol=1e-9, max_iter=60):
    """Numerically invert ANY monotonic non-decreasing CDF via bisection: find x with
    F(x) = target_prob. This is the quantile function Q(p), computed with no closed form."""
    for _ in range(max_iter):
        mid = (lo + hi) / 2
        if cdf_func(mid) < target_prob:
            lo = mid
        else:
            hi = mid
        if hi - lo < tol:
            break
    return (lo + hi) / 2


# Cross-check our generic bisection inverter against scipy's closed-form-backed norm.ppf
print("\n--- Cross-checking quantile_from_cdf against scipy.stats.norm.ppf ---")
for target_prob in [0.1, 0.25, 0.5, 0.75, 0.9]:
    ours = quantile_from_cdf(norm.cdf, target_prob, lo=-10, hi=10)
    scipy_answer = norm.ppf(target_prob)
    print(f"  p={target_prob}: ours={ours:.6f}, scipy={scipy_answer:.6f}, "
          f"diff={abs(ours - scipy_answer):.2e}")

# Inverse transform sampling FROM SCRATCH: if U~Uniform(0,1), then Q(U) has distribution F.
# We sample from an Exponential(rate=1.5) using only our bisection-based quantile function.
rng = np.random.default_rng(0)
rate = 1.5
exp_cdf = lambda x: 1 - np.exp(-rate * x)  # closed-form CDF, used only as the function we invert

n_samples = 3000
uniforms = rng.uniform(0, 1, size=n_samples)
samples_from_scratch = np.array([quantile_from_cdf(exp_cdf, u, lo=0, hi=30) for u in uniforms])

print(f"\n--- Inverse transform sampling from Exponential(rate={rate}) via our quantile_from_cdf ---")
print(f"Sample mean: {samples_from_scratch.mean():.4f}  (theoretical E[X]=1/rate={1/rate:.4f})")
print(f"Sample std:  {samples_from_scratch.std():.4f}  (theoretical std=1/rate={1/rate:.4f})")

# Independent cross-check: compare our samples' histogram to scipy's own Exponential PDF
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(samples_from_scratch, bins=40, density=True, alpha=0.6, color="#4C72B0", label="our samples")
x_plot = np.linspace(0, 6, 300)
ax.plot(x_plot, expon(scale=1 / rate).pdf(x_plot), "k--", lw=2, label="true PDF (scipy)")
ax.set_xlabel("x"); ax.set_ylabel("density"); ax.legend()
ax.set_title("Inverting our OWN quantile function reproduces the target PDF")
plt.tight_layout()
plt.show()

## 🤖 Why This Matters for AI

Every softmax layer in every classifier and every transformer's output head produces a
**categorical PMF** over classes or vocabulary tokens — the exact object this notebook defines.
And understanding a *feature's* distribution (not just its raw values) is what makes principled
preprocessing possible: quantiles tell you where the "typical" and "extreme" values actually
are, rather than guessing.

Specifically, `torch.nn.Softmax(dim=-1)` converts raw logits to a categorical PMF, and `torch.distributions.Categorical` wraps it into a proper distribution object with `.sample()`, `.log_prob()`, and `.entropy()` methods — the exact operations defined in this notebook, packaged for production use.

---

In [ ]:
# Part 1: softmax converts arbitrary real-valued logits into a valid categorical PMF over tokens
def softmax(logits):
    shifted = logits - np.max(logits)  # numerically stable: subtract max before exponentiating
    exps = np.exp(shifted)
    return exps / exps.sum()

rng = np.random.default_rng(0)
vocab_size = 20
logits = rng.normal(scale=2.0, size=vocab_size)   # a toy "language model output layer"
token_probs = softmax(logits)

print("Softmax output is a valid PMF over the vocabulary (the axioms from 03.01, checked directly):")
print(f"  all probs >= 0? {np.all(token_probs >= 0)}")
print(f"  sum(probs) = {token_probs.sum():.10f}  (must be exactly 1)")

# The CDF over probability-SORTED tokens answers: how many tokens hold 90% of the total mass?
sorted_probs_desc = np.sort(token_probs)[::-1]
cumulative_mass = np.cumsum(sorted_probs_desc)
n_tokens_for_90pct = np.searchsorted(cumulative_mass, 0.9) + 1
print(f"\nOut of {vocab_size} tokens, the top {n_tokens_for_90pct} account for 90% of the probability mass")
print("(this cumulative-mass view of a PMF, applied at generation time, is exactly nucleus/top-p")
print(" sampling -- we build the full decoding algorithm in 03.09)")

# Part 2: modeling a feature's distribution with scipy.stats, then using ITS quantile function
# (.ppf) for principled outlier clipping -- a real preprocessing decision, not an arbitrary one
activations = rng.normal(loc=5.0, scale=2.0, size=10_000)  # e.g. pre-activation values of a layer
fitted_mu, fitted_sigma = norm.fit(activations)  # MLE fit, derived properly in 03.07
print(f"\nFitted feature distribution: N(mu={fitted_mu:.3f}, sigma={fitted_sigma:.3f})  (true: N(5, 2))")

fitted_rv = norm(fitted_mu, fitted_sigma)
lower_clip, upper_clip = fitted_rv.ppf([0.01, 0.99])  # 1st/99th percentile via the quantile function
frac_outside_clip = np.mean((activations < lower_clip) | (activations > upper_clip))
print(f"1st/99th percentile clip range from .ppf: [{lower_clip:.3f}, {upper_clip:.3f}]")
print(f"Fraction of activations actually outside this range: {frac_outside_clip:.4f}  (expect ~0.02, i.e. 2%)")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For a fair six-sided die ($X\in\{1,\ldots,6\}$, uniform PMF), compute $F_X(3.5)$ and
   $F_X(6)$ by hand. As a sanity check, confirm $F_X$ is non-decreasing across
   $x=0,1,3.5,6,10$ and that $F_X(6)=1$ exactly (the CDF must reach 1 at the top of the
   support). Verify with `scipy.stats.randint(1, 7).cdf(...)`.

<details>
<summary>💡 Solution</summary>

$F_X(3.5) = P(X\le 3.5) = P(X\in\{1,2,3\}) = 3/6 = 0.5$ (the die takes integer values, so
everything up to and including 3 counts, and $3.5$ doesn't include 4). $F_X(6) = P(X\le 6) =
6/6 = 1$, since every outcome is $\le 6$. Checking monotonicity: $F_X(0)=0 \le F_X(1)=1/6 \le
F_X(3.5)=0.5 \le F_X(6)=1 \le F_X(10)=1$ — non-decreasing throughout, and it correctly saturates
at exactly $1$ from $x=6$ onward rather than continuing to grow, confirming normalization.
`scipy.stats.randint(1, 7).cdf(3.5)` gives `0.5` and `.cdf(6)` gives `1.0`, matching both by-hand
values exactly.

</details>

### 💭 UNDERSTAND
2. The notebook states a PDF "is not a probability" and "can exceed 1." $\mathcal{N}(0, 1)$ has
   peak density $f(0)\approx 0.399 < 1$, but $\mathcal{N}(0, 0.1)$ (same mean, ten times smaller
   standard deviation) has peak density $f(0)\approx 3.989 > 1$. Explain why shrinking the
   standard deviation *must* increase the peak density, using only the normalization constraint
   $\int f(x)\,dx=1$ — no computation of the exact value required.

<details>
<summary>💡 Solution</summary>

The total area under any PDF must equal exactly $1$, by normalization. A smaller standard
deviation concentrates $\mathcal{N}(\mu,\sigma)$'s probability into a narrower interval around
the mean — the curve's "width" shrinks roughly proportionally to $\sigma$. To keep the *area*
under a narrower curve equal to the same fixed value ($1$), the curve's *height* must grow
correspondingly — informally, area $\approx$ height $\times$ width, so if width shrinks by a
factor of $10$ (as it does from $\sigma=1$ to $\sigma=0.1$), height must grow by roughly that
same factor to keep the product fixed. This is exactly why the peak density scales as $1/\sigma$
for a Gaussian (and why a density, unlike a probability, has no upper bound of $1$ — only its
integral does).

</details>

### ✏️ DERIVE
3. Derive the median of an Exponential distribution with rate $\lambda$ (CDF
   $F(x) = 1-e^{-\lambda x}$ for $x\ge 0$) analytically, by solving $F(x)=0.5$ for $x$. Then, for
   $\lambda=1.5$, confirm your closed-form answer matches both `quantile_from_cdf` (bisecting the
   notebook's own `exp_cdf`) and `scipy.stats.expon(scale=1/1.5).ppf(0.5)`.

<details>
<summary>💡 Solution outline</summary>

Set $F(x) = 0.5$: $1 - e^{-\lambda x} = 0.5 \implies e^{-\lambda x} = 0.5 \implies -\lambda x =
\ln(0.5) = -\ln 2 \implies x = \dfrac{\ln 2}{\lambda}$. For $\lambda = 1.5$:
$\text{median} = \ln(2)/1.5 \approx 0.4621$. Both `quantile_from_cdf(exp_cdf, 0.5, lo=0, hi=30)`
and `expon(scale=1/1.5).ppf(0.5)` evaluate to $0.462098\ldots$, matching the closed form to
numerical precision — confirming the bisection-based inverter recovers the exact analytic
answer without ever being told the formula.

</details>

### 🐍 IMPLEMENT
4. Implement a generic finite-difference PDF estimator `fd_pdf(cdf_func, x, h=1e-5)` that
   approximates $f_X(x) = F_X'(x)$ via the central difference
   $\big(F(x+h)-F(x-h)\big)/(2h)$ (recall the derivative definition from `02.01`–`02.03`),
   and check it against `scipy.stats.norm.pdf` and `scipy.stats.expon(scale=1/1.5).pdf` at
   several points.

<details>
<summary>✅ How to know you're right</summary>

`fd_pdf(norm.cdf, x)` should match `norm.pdf(x)` to within about `1e-10` for `x` in
$\{-1, 0, 0.5, 1.5\}$, and `fd_pdf(expon(scale=1/1.5).cdf, x)` should match
`expon(scale=1/1.5).pdf(x)` to similar precision for `x` in $\{0.1, 0.5, 1.0, 2.0\}$ — the
finite-difference derivative of any CDF should recover its PDF, since that relationship
($f=F'$) is exact, not approximate; the only error should be the small, controllable
discretization error from using a finite `h` rather than a true limit.

</details>

### 🤖 APPLY
5. Modify the softmax temperature by computing `softmax(logits / T)` for
   $T \in \{0.5, 1.0, 2.0\}$, using the notebook's own `logits` array. For each $T$, recompute
   how many tokens are needed to cover 90% of the probability mass, and also record the single
   largest token probability. Explain the trend, and connect it to why a language model's
   sampling "temperature" is described as controlling how "confident" or "random" its next-token
   choice is (fully implemented in `03.09`).

<details>
<summary>✅ What you should observe</summary>

With this notebook's own seeded `logits`, the number of tokens needed for 90% of the mass goes
$3\to 9\to 14$ as $T$ goes $0.5\to 1.0\to 2.0$, while the single largest probability drops
correspondingly ($0.573\to 0.313\to 0.160$) — every softmax output still sums to exactly $1$
regardless of $T$ (normalization is temperature-invariant), but dividing logits by a smaller $T$
*before* exponentiating stretches apart the gaps between them, concentrating mass onto fewer,
larger-logit tokens ("confident"/peaked), while a larger $T$ compresses the gaps, spreading mass
more evenly across many tokens ("random"/flat). This is exactly the mechanism `03.09`'s
temperature-based decoding relies on.

</details>

### 🚀 CHALLENGE
6. The **probability integral transform** says that if $X$ has continuous CDF $F$, then
   $F(X)$ is distributed exactly Uniform$(0,1)$ — regardless of what $F$ is. This gives a
   general way to check whether a *fitted* model's CDF actually matches the data it was fit to:
   apply the model's CDF to held-out data and check whether the result looks Uniform$(0,1)$.
   Generate 5000 samples from a true $\mathcal{N}(5, 2)$, then compute the transform
   $F_{\text{model}}(\text{data})$ under three candidate models: (a) the correct
   $\mathcal{N}(5,2)$, (b) a wrong-scale $\mathcal{N}(5, 0.8)$, and (c) a wrong-location
   $\mathcal{N}(6.5, 2)$. For each, compare the transformed values' histogram to Uniform$(0,1)$
   (visually, and with `scipy.stats.kstest(transformed, 'uniform')`). Characterize *how* the
   histogram's shape distorts differently for a wrong-scale vs. a wrong-location model.

<details>
<summary>🔍 What a strong answer covers</summary>

Under the correct model (a), the Kolmogorov-Smirnov test against Uniform$(0,1)$ should NOT
reject (in a representative run, a KS statistic around $0.01$–$0.02$ with a large p-value), and
the histogram of transformed values should look flat. Both misspecified models are decisively
rejected (KS statistic around $0.2$–$0.3$, p-value effectively $0$) but with qualitatively
different histogram shapes: the wrong-scale model (b), whose spread is too narrow, pushes mass
in the true data's tails toward the *extremes* $0$ and $1$ (the badly-fit model calls
true-data tail points "impossibly extreme" under its own too-narrow distribution) producing a
U/bathtub-shaped histogram, while the wrong-location model (c), shifted off-center, produces a
histogram skewed toward one end (values systematically too high or too low) rather than
U-shaped, since a location shift moves the whole transform in one direction rather than
stretching its tails. A strong answer explicitly distinguishes these two distortion *shapes*
rather than only reporting that both models fail the test, and notes the general applicability
of this diagnostic: it is exactly the model-checking tool used to validate that a fitted
probabilistic model (any model in `03.07`, `03.10`, or beyond) is well-calibrated, not just
"close" by eye.

</details>

---

## 📚 Further Reading
- Wasserman, *All of Statistics*, Ch. 2 (Random Variables)
- `scipy.stats` [documentation](https://docs.scipy.org/doc/scipy/reference/stats.html) — the continuous/discrete distribution API used throughout this module
- Devroye, [*Non-Uniform Random Variate Generation*](http://www.nrbook.com/devroye/) (inverse transform sampling, Ch. 2)

---

*Next notebook: [Expectation, Variance, and Moments](03_expectation_variance_moments.ipynb)*